## Orchestration

In [1]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
sg = SendGridAPIClient(os.environ.get('SENDGRID_API_KEY'))

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

In [3]:
message = Mail(
    from_email='kavyahj26@gmail.com',  # Must match your verified sender/domain
    to_emails='khj@binghamton.edu',
    subject='Testing SendGrid Integration',
    html_content='<strong>It works!</strong> Your SendGrid API integration is complete.'
)

In [4]:
def send_email(message):
    try:
        response = sg.send(message)
        print(f"Success! Status Code: {response.status_code}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
send_email(message)

## Agent Orchestration

### https://openai.github.io/openai-agents-python/multi_agent/

## 1. Orchestrating by Code

In [20]:
intro = """ You are an expert career strategist and student outreach agent. Your goal is to draft highly tailored, high-converting cold outreach emails to recruiters, hiring managers, or team leads.
Task: Analyze the provided job description, identify key business challenges or team needs, and compose a concise, professional outreach email demonstrating how I (the student) can add immediate value.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous and just 2 lines"
instructions3 = intro + "Your email style is punchy, high-impact, and unapologetically confident—framing me as a top-tier candidate who brings unmatched value to the team."

In [21]:
student_agent1 = Agent(name = "Professional Student", instructions=instructions1, model = MODEL_NAME)
student_agent2 = Agent(name = "Professional Student", instructions=instructions2, model = MODEL_NAME)
student_agent3 = Agent(name = "Professional Student", instructions=instructions3, model = MODEL_NAME)

In [10]:
job_description = """Sierra is a pioneering platform dedicated to transforming customer interactions through advanced AI agents. Serving some of the world's most prominent brands, including The GAP, Rocket Mortgage, SoFi, Sutter Health, and SoftBank, Sierra specializes in creating intelligent solutions that enhance customer service and drive business growth. The company operates primarily from its headquarters in San Francisco and has expanding offices across North America, Europe, and Asia, reflecting its global reach and commitment to innovation.

Guided by core values such as Trust, Customer Obsession, Craftsmanship, Intensity, and a commitment to work-life balance, Sierra fosters a culture of excellence, integrity, and collaboration. Its leadership team includes industry veterans like Bret Taylor, Board Chair of OpenAI and former co-CEO of Salesforce, and Clay Bavor, a veteran at Google leading initiatives in AR/VR and productivity tools. These leaders drive Sierra's vision to develop cutting-edge AI solutions that redefine customer engagement across various industries.

At Sierra, the focus is on building scalable, reliable, and impactful AI agents that empower organizations to deliver personalized, efficient, and memorable customer experiences. The company's innovative approach combines deep technological expertise with a customer-centric mindset, ensuring that its solutions are not only powerful but also aligned with client needs and industry standards.

About The Role

Sierra is seeking a talented and motivated AI Engineer to join its Tech, Media & Telecom team. This role offers a unique opportunity to work at the forefront of AI development, building production-grade AI agents that serve as critical components in enhancing customer interactions for some of the world’s leading brands. The successful candidate will be instrumental in designing, developing, and deploying scalable AI systems that directly contribute to Sierra’s growth and innovation objectives.

In this position, you will have ownership over the entire Agent Development Life Cycle, from initial pilot phases to full deployment and ongoing refinement. You will collaborate closely with enterprise clients and startups alike, translating complex business challenges into effective AI solutions. Additionally, your insights and work will influence the evolution of Sierra’s core platform, helping to identify unmet needs, prototype new features, and shape the future of AI agent technology.

This role requires a proactive, creative, and technically skilled individual who thrives in dynamic environments and is passionate about leveraging AI to solve real-world problems. You will be part of a collaborative team that values craftsmanship, innovation, and customer focus, contributing to projects that impact millions of users worldwide.

Qualifications

To excel in this role, candidates should possess a strong background in building and scaling end-to-end production systems. Demonstrated experience with AI, machine learning, or related fields is essential, along with proficiency in relevant programming languages such as Python, React, TypeScript, or Go. The ideal candidate will have excellent problem-solving skills, especially in fast-paced, ambiguous environments, and a strong desire to innovate and experiment.

Effective communication skills are critical, as you will frequently interact with both technical and non-technical stakeholders. A builder’s mindset, high agency, and the ability to work independently are highly valued. Previous experience in deploying AI/LLM systems in production environments, working with enterprise clients, or leading technical projects will be advantageous.

Additional preferred qualifications include familiarity with tools such as eval frameworks, agent tooling, RAG pipelines, prompt engineering, and experience with voice or conversational AI. Prior entrepreneurial experience or roles involving customer engagement and stakeholder management are also considered a plus.

Responsibilities

Design and deliver production-grade AI agents that are reliable, scalable, and intuitive, directly contributing to Sierra’s growth and customer success. You will be responsible for building high-performance systems that handle complex customer interactions across various industries such as finance, healthcare, and commerce.

Lead the entire Agent Development Life Cycle, from initial pilot testing to deployment, tuning, and continuous iteration. Define and implement best practices for development, deployment, and maintenance of AI agents, ensuring quality and efficiency at every stage.

Collaborate with large enterprises and innovative startups to understand their unique business challenges and craft tailored AI solutions. Your work will help transform customer engagement strategies and operational workflows at scale.

Contribute to the evolution of Sierra’s core platform by surfacing unmet needs, prototyping new features, and working cross-functionally with research, product, and platform teams. Your insights will help shape the future of Sierra’s AI agent capabilities and overall product roadmap.

Engage in projects such as developing goal-oriented agents, personalization systems powered by Sierra’s Context Engine, conversational discovery tools, and deep troubleshooting agents for telecommunications and connected devices. Experiment with voice experiences and tailored Voice Personas to enhance brand-specific customer interactions.

Benefits

Sierra offers a comprehensive benefits package designed to support the well-being and growth of its employees. Full-time team members enjoy flexible, unlimited paid time off, enabling work-life balance. The company provides medical, dental, and vision coverage for employees and their families, along with life insurance and disability benefits.

Retirement plans are available depending on the country of employment, complemented by parental leave policies and fertility and family-building benefits through partners like Carrot. Employees also receive daily meals, snacks, and coffee to foster a productive environment, along with a discretionary benefit stipend to spend on personal needs or interests.

Additional perks include opportunities for professional development and community engagement, all within a culture that emphasizes respect, support, and recognition of individual achievements. Sierra’s benefits are aligned with its core values and adaptable to regional requirements, with participation in equity plans available for eligible employees
"""

In [22]:
user = """I'm Kavya Hosamane Jayanna, a Full Stack AI Engineer (RAG · LLM · .NET · React · AWS) with an MS in Computer Science from Binghamton University and a BE in Electronics and Communication from MVJ College of Engineering, Bangalore. I'm AWS Certified Cloud Practitioner. My background includes 2+ years as a Software/Senior Software Engineer at Happiest Minds Technologies, plus research assistant roles in Generative AI, multimodal research, and a Product Design & Data Analytics internship at Neuro20 Technologies. My skills span Python, .NET, React, Machine Learning, Deep Learning (PyTorch, TensorFlow, CNNs), LangChain, Vector Embeddings/RAG systems, Hugging Face, AWS, and SQL.

LinkedIn: https://linkedin.com/in/kavya-h-j
GitHub: https://github.com/kavyahj04
Portfolio: https://my-porttfolio-lemon.vercel.app/
"""

In [23]:
user_input = "Hi I want to apply to this job---" +"\n" + job_description + "\n" + "This is my information -----   " + user + "\n" + "Please help me to generate a mail"

In [ ]:
result = Runner.run_streamed(student_agent1, input=user_input)
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
with trace("Parallel Cold Emails"):
    results = await asyncio.gather(
        Runner.run(student_agent1, input=user_input),
        Runner.run(student_agent2, input=user_input),
        Runner.run(student_agent3, input=user_input),
    )

    outputs = [result.final_output for result in results]

    for out in outputs:
        print(out + "\n\n")

In [29]:
decision = """
You pick the best email from the given options that matches student description the best.
Do not give an explanation; reply with the selected email only.
"""

email_picker = Agent(name="Email_Picker", instructions=decision, model=MODEL_NAME)

In [ ]:
with trace("Email Workflow"):
    results = await asyncio.gather(
        Runner.run(student_agent1, user_input),
        Runner.run(student_agent2, user_input),
        Runner.run(student_agent3, user_input)
    )

    outputs = [result.final_output for result in results]
    emails = "Cold emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(email_picker, emails)
    print(f"Best Student email:\n{best.final_output}")

In [ ]:
message = Mail(
    from_email='kavyahj26@gmail.com',  # Must match your verified sender/domain
    to_emails='khj@binghamton.edu',
    subject='Testing SendGrid Integration',
    html_content='<strong>It works!</strong> Your SendGrid API integration is complete.'
)

In [32]:
@function_tool
def send_email_tool(subject: str, html_body: str) -> str:
    """Send an email to the recipient with the given subject and HTML body.

    Args:
        subject: The subject line of the email
        html_body: The body of the email as HTML
    """
    message = Mail(
        from_email='kavyahj26@gmail.com',  # Must match your verified sender/domain
        to_emails='khj@binghamton.edu',
        subject=subject,
        html_content=html_body,
    )
    send_email(message)
    return "Email sent successfully"


In [ ]:
send_email_tool.params_json_schema

In [36]:
decision = """
You pick the best email from the given options to help a student to reach out to company.
Imagine you are a HR and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

email_sender = Agent(name="Email Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
with trace("Send mail to Kavya"):
    results = await asyncio.gather(
        Runner.run(student_agent1, input=user_input),
        Runner.run(student_agent2, input=user_input),
        Runner.run(student_agent3, input=user_input)
    )
    outputs = [result.final_output for result in results]

    emails = "Emails :\n\n" + "\n\nEmail:\n\n".join(outputs)

    best_picked = await Runner.run(email_sender, emails)

    print(f"Final response \n\n{best_picked.final_output}")


## Orchestrating by LLMs


### 3a. Via tools

In [41]:
description = "Use this tool to write a reach out email. In the input, just instruct it to write a email."
tool1 = student_agent1.as_tool(tool_name="student_email_writer_1", tool_description=description)
tool2 = student_agent2.as_tool(tool_name="student_email_writer_1", tool_description=description)
tool3 = student_agent3.as_tool(tool_name="student_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]
